<a href="https://colab.research.google.com/github/ahmadisyraf39/palm-oil-insights/blob/main/palm_oil_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Malaysian Palm Oil Price & Production Insights**

Do rainfall and production affect palm oil prices, and can next month's price be forecast? This notebook collects, cleans and analyses public data from the World Bank, MPOB and Open-Meteo.

**Step 1: Setup**

Connect Google Drive to access the data files, and import pandas for data handling and requests for downloading data from APIs.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

folder = "/content/drive/MyDrive/palm_oil_project/"

import pandas as pd
import requests
print("Ready!", pd.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready! 2.2.3


**Step 2: Palm oil prices**

Load monthly palm oil prices (USD/tonne) from the World Bank Pink Sheet, clean the dates, and keep data from 2015 onward.

In [2]:
f = folder + "CMO-Historical-Data-Monthly.xlsx"

# 1. Read the sheet with no headers so we can find which row holds the column names
raw_price = pd.read_excel(f, sheet_name="Monthly Prices", header=None)

# 2. Find the row number that contains the text "Palm oil"
hdr = raw_price.index[raw_price.apply(lambda r: r.astype(str).str.contains("Palm oil").any(), axis=1)][0]
print("Header row is:", hdr)

# 3. Read the sheet again, using that row as column names
price = pd.read_excel(f, sheet_name="Monthly Prices", header=hdr)
price.columns = price.columns.astype(str).str.strip()   # remove stray spaces from names

# 4. Keep only the date column and the palm oil column
price = price.rename(columns={price.columns[0]: "month"})
price = price[["month", "Palm oil"]]

# 5. Keep only rows whose month looks like "2015M01"
price = price[price["month"].astype(str).str.match(r"\d{4}M\d{2}")].copy()

# 6. Turn "2015M01" into a real date: "2015-01-01"
price["date"] = pd.to_datetime(price["month"].str.replace("M", "-") + "-01")

# 7. Tidy up: rename the column and make sure prices are numbers
price = price[["date", "Palm oil"]].rename(columns={"Palm oil": "price_usd"})
price["price_usd"] = pd.to_numeric(price["price_usd"], errors="coerce")

# 8. Keep 2015 onward
price = price[price["date"] >= "2015-01-01"]

# print the first (n) rows
display(price.head(10))

# print number of rows and columns
print(price.shape)

# print the last (n) rows
display(price.tail())

Header row is: 4


,date,price_usd
661,2015-01-01,720
662,2015-02-01,723
663,2015-03-01,699
664,2015-04-01,683
665,2015-05-01,697
666,2015-06-01,706
667,2015-07-01,680
668,2015-08-01,601
669,2015-09-01,600
670,2015-10-01,637


(140, 2)


,date,price_usd
796,2026-04-01,1146
797,2026-05-01,1136
798,2026-06-01,1104
799,2026-07-01,1101
800,2026-08-01,1117


**Step 3: Rainfall**

Download daily rainfall for Sandakan, Sabah, from the Open-Meteo API and sum it per month, since palm oil yields respond to rainfall with a delay.

In [3]:
url = ("https://archive-api.open-meteo.com/v1/archive"
       "?latitude=5.84&longitude=118.12"          # Sandakan, Sabah (a major palm oil area)
       "&start_date=2015-01-01&end_date=2026-08-31"
       "&daily=precipitation_sum"                  # total rain per day, in mm
       "&timezone=Asia/Kuala_Lumpur")

response = requests.get(url)       # ask the API for data
d = response.json()["daily"]       # JSON is a common data format; take the "daily" part

rain = pd.DataFrame({"date": pd.to_datetime(d["time"]), "rain_mm": d["precipitation_sum"]})
print(rain.shape)
display(rain.head())

rain = rain.resample("MS", on="date").sum().reset_index()
print(rain.shape)
display(rain.head())

(4261, 2)


,date,rain_mm
0,2015-01-01,49.3
1,2015-01-02,39.7
2,2015-01-03,9.4
3,2015-01-04,8.5
4,2015-01-05,4.6


(140, 2)


,date,rain_mm
0,2015-01-01,388.9
1,2015-02-01,91.2
2,2015-03-01,75.3
3,2015-04-01,139.3
4,2015-05-01,120.0


**Step 4: Merge and check**

Combine prices and rainfall by month, check for missing values, and save the clean dataset.

In [4]:
df = price.merge(rain, on="date")
print(df.shape)
display(df.head())

print(df.isna().sum())         # count missing values per column
print(df["date"].min(), "to", df["date"].max())
display(df.describe())                  # min, max, average of each column

df.to_csv(folder + "palm_oil_prices_and_rainfall.csv", index=False)

(140, 3)


,date,price_usd,rain_mm
0,2015-01-01,720,388.9
1,2015-02-01,723,91.2
2,2015-03-01,699,75.3
3,2015-04-01,683,139.3
4,2015-05-01,697,120.0


date         0
price_usd    0
rain_mm      0
dtype: int64
2015-01-01 00:00:00 to 2026-08-01 00:00:00


,date,price_usd,rain_mm
count,140,140.000000,140.000000
mean,2020-10-15 19:53:08.571428608,868.664286,257.557857
min,2015-01-01 00:00:00,535.000000,50.200000
25%,2017-11-23 12:00:00,696.250000,168.650000
50%,2020-10-16 12:00:00,815.500000,251.800000
75%,2023-09-08 12:00:00,1007.750000,330.950000
max,2026-08-01 00:00:00,1777.000000,670.400000
std,NaN,243.262082,117.032303


**Step 5: MPOB production and stocks**

Add Malaysian crude palm oil production and total palm oil closing stocks from MPOB, matched by month. Gaps in the production series are backfilled from MPOB's yearly Summary reports.

In [5]:
folder = "/content/drive/MyDrive/palm_oil_project/"

# MPOB monthly data via palmoileconomics.com (long format: one row per month per indicator)
raw = pd.read_csv(folder + "mpob_raw.csv")

print(raw.shape)
print(raw["indicator"].unique())
display(raw.head())

(1694, 6)
['cpo_stock' 'processed_po_stock' 'total_po_stock' 'cpko_production'
 'cpo_import' 'cpo_production' 'pk_production' 'pkc_production'
 'pko_stock' 'po_export' 'po_import' 'ffb_price' 'biodiesel_export'
 'oleochemical_export' 'pkc_export' 'pko_export']


,month,indicator,value,unit,revision,source
0,2000-01-01,cpo_stock,629381.0,tonnes,bepi,Malaysian Palm Oil Board (MPOB) monthly indust...
1,2000-01-01,processed_po_stock,330481.0,tonnes,bepi,Malaysian Palm Oil Board (MPOB) monthly indust...
2,2000-01-01,total_po_stock,959862.0,tonnes,bepi,Malaysian Palm Oil Board (MPOB) monthly indust...
3,2000-02-01,cpo_stock,574161.0,tonnes,bepi,Malaysian Palm Oil Board (MPOB) monthly indust...
4,2000-02-01,processed_po_stock,333590.0,tonnes,bepi,Malaysian Palm Oil Board (MPOB) monthly indust...


In [6]:
# Keep the 2 indicators and reshape to one row per month
mpob = raw[raw["indicator"].isin(["cpo_production", "total_po_stock"])]
mpob = mpob.pivot_table(index="month", columns="indicator", values="value", aggfunc="last").reset_index()
mpob = mpob.rename(columns={"month": "date", "cpo_production": "production_t", "total_po_stock": "stocks_t"})
mpob["date"] = pd.to_datetime(mpob["date"])
mpob = mpob[mpob["date"] >= "2015-01-01"]

print("Missing production months:", mpob["production_t"].isna().sum())
display(mpob.tail())

Missing production months: 76


indicator,date,production_t,stocks_t
315,2026-04-01,1629801.0,2308944.0
316,2026-05-01,1516327.0,2427943.0
317,2026-06-01,1638796.0,2543830.0
318,2026-07-01,1792596.0,2627898.0
319,2026-08-01,1817499.0,2824488.0


In [7]:
fill = pd.read_csv(folder + "production_backfill.csv", parse_dates=["date"])

mpob = mpob.set_index("date")
mpob["production_t"] = mpob["production_t"].fillna(fill.set_index("date")["production_t"])
mpob = mpob.reset_index()

print("Missing production months after backfill:", mpob["production_t"].isna().sum())
mpob.to_csv(folder + "mpob_production.csv", index=False)

Missing production months after backfill: 0


In [8]:
# Drop MPOB columns if this cell was run before, so rerunning never creates _x/_y duplicates
df = df.drop(columns=["production_t", "stocks_t"], errors="ignore")

df = df.merge(mpob, on="date", how="left")
df["rain_mm"] = df["rain_mm"].round(1)

print(df.shape)
print(df.isna().sum())
df.to_csv(folder + "palm_oil_monthly.csv", index=False)
display(df.tail())

(140, 5)
date            0
price_usd       0
rain_mm         0
production_t    0
stocks_t        0
dtype: int64


,date,price_usd,rain_mm,production_t,stocks_t
135,2026-04-01,1146,50.2,1629801.0,2308944.0
136,2026-05-01,1136,143.5,1516327.0,2427943.0
137,2026-06-01,1104,234.5,1638796.0,2543830.0
138,2026-07-01,1101,174.1,1792596.0,2627898.0
139,2026-08-01,1117,106.8,1817499.0,2824488.0
